# Descarga de imágenes desde links de Excel — EPII / SAVIIA

Este notebook lee los links del Excel de evaluación e intenta descargar las imágenes de Google Drive



## 1. Setup


In [ ]:
!pip -q install pandas openpyxl requests tqdm beautifulsoup4 playwright nest_asyncio
!python -m playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 18.5 MB/s eta 0:00:00
177 MiB [] 0% 364.7s177 MiB [] 0% 261.6s177 MiB [] 0% 266.7s177 MiB [] 0% 884.7s177 MiB [] 0% 731.4s177 MiB [] 0% 993.0s177 MiB [] 0% 876.4s177 MiB [] 0% 791.6s177 MiB [] 0% 729.3s177 MiB [] 0% 679.7s177 MiB [] 0% 639.9s177 MiB [] 0% 630.1s177 MiB [] 0% 606.7s177 MiB [] 0% 582.6s177 MiB [] 0% 560.4s177 MiB [] 0% 539.6s177 MiB [] 0% 519.9s177 MiB [] 0% 502.6s177 MiB [] 0% 485.3s177 MiB [] 0% 469.3s177 MiB [] 0% 455.4s177 MiB [] 0% 423.7s177 MiB [] 0% 398.6s177 MiB [] 0% 376.6s177 MiB [] 0% 358.5s177 MiB [] 0% 342.1s177 MiB [] 0% 327.7s177 MiB [] 0% 315.4s177 MiB [] 0% 304.1s177 MiB [] 0% 292.8s177 MiB [] 0% 283.4s177 MiB [] 0% 269.1s177 MiB [] 0% 256.1s177 MiB [] 0% 244.5s177 MiB [] 0% 234.1s177 MiB [] 0% 221.3s177 MiB [] 0% 210.0s177 MiB [] 0% 199.8s177 MiB [] 0% 189.6s177 MiB [] 0% 181.6s177 MiB [] 0% 174.5s177 MiB [] 0% 168.0s177 MiB [] 0% 160.3s177 MiB [] 0% 153.5s177 MiB [] 0% 147.3s177 MiB [] 0% 140.5s1

In [ ]:
import os
import re
import json
import time
import mimetypes
import asyncio
import unicodedata
from pathlib import Path
from urllib.parse import urlparse, parse_qs, unquote

import nest_asyncio
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

nest_asyncio.apply()

## 2. Configuración

El Excel subido tiene dos hojas principales: `Ground Truth Dataset` y `Plantilla de Evaluación`. Ambas tienen una columna `Link`. Por defecto se usa `Plantilla de Evaluación`, pero puedes cambiarlo.

In [ ]:
# Rutas
USE_GOOGLE_DRIVE = True
EXCEL_PATH = "/content/drive/MyDrive/ECHO/Protocolos/00_Evaluacion_SAVIIA_ML_Edge/Dataset_Evaluacion_EPII.xlsx"
SHEET_NAME = "Plantilla de Evaluación"
LINK_COLUMN = "Link"
# Carpeta de salida.
OUTPUT_DIR = Path("/content/drive/MyDrive/ECHO/Metadata/downloaded_eval_original_images")
# Opciones de descarga
OVERWRITE = False
MAX_ROWS = None 
REQUEST_SLEEP_SEC = 0.25
PREFERRED_THUMBNAIL_SIZES = [4096, 3072, 2048, 1600, 1024]
# Fallback por navegador headless para inspeccionar la preview de Drive.
USE_PLAYWRIGHT_FALLBACK = True
HEADLESS_BROWSER = True
BROWSER_TIMEOUT_MS = 60_000

## 3. Montar Drive y leer Excel

In [ ]:
# Montaje de Drive y preparación de paths
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)

OUTPUT_DIR: /content/drive/MyDrive/ECHO/Metadata/downloaded_eval_original_images


In [ ]:
excel_path = Path(EXCEL_PATH)

xls = pd.ExcelFile(excel_path)
print("Excel:", excel_path)
print("Hojas disponibles:", xls.sheet_names)

df = pd.read_excel(excel_path, sheet_name=SHEET_NAME)
print("Filas:", len(df))
print("Columnas:", list(df.columns))
display(df.head())

Excel: /content/drive/MyDrive/ECHO/Protocolos/00_Evaluacion_SAVIIA_ML_Edge/Dataset_Evaluacion_EPII.xlsx
Hojas disponibles: ['Ground Truth Dataset', 'Plantilla de Evaluación']
Filas: 77
Columnas: ['Link', 'Camera ID', '¿Detectas un animal claro?', 'Especie (Clasificación)']


,Link,Camera ID,¿Detectas un animal claro?,Especie (Clasificación)
0,https://drive.google.com/file/d/1IZAOIR8S5BK35...,CAM6,NaN,NaN
1,https://drive.google.com/file/d/1umhS6rmaguIBs...,CAM1,NaN,NaN
2,https://drive.google.com/file/d/1q6xDb6qiuxyRN...,CAM6,NaN,NaN
3,https://drive.google.com/file/d/1lysghJxxHy-RF...,CAM2,NaN,NaN
4,https://drive.google.com/file/d/1ev8j9iS1w6VUX...,CAM4,NaN,NaN


In [ ]:
if LINK_COLUMN not in df.columns:
    raise ValueError(f"No existe la columna '{LINK_COLUMN}'. Columnas disponibles: {list(df.columns)}")

work_df = df.copy()
work_df = work_df[work_df[LINK_COLUMN].notna()].reset_index(drop=True)

if MAX_ROWS is not None:
    work_df = work_df.head(MAX_ROWS).copy()

print("Links a procesar:", len(work_df))
display(work_df.head())

Links a procesar: 77


,Link,Camera ID,¿Detectas un animal claro?,Especie (Clasificación)
0,https://drive.google.com/file/d/1IZAOIR8S5BK35...,CAM6,NaN,NaN
1,https://drive.google.com/file/d/1umhS6rmaguIBs...,CAM1,NaN,NaN
2,https://drive.google.com/file/d/1q6xDb6qiuxyRN...,CAM6,NaN,NaN
3,https://drive.google.com/file/d/1lysghJxxHy-RF...,CAM2,NaN,NaN
4,https://drive.google.com/file/d/1ev8j9iS1w6VUX...,CAM4,NaN,NaN


## 4. Funciones auxiliares

In [ ]:
def extract_drive_file_id(url: str):
    """Extrae file_id desde links típicos de Google Drive."""
    if not isinstance(url, str) or not url.strip():
        return None

    url = url.strip()
    patterns = [
        r"/file/d/([^/]+)",
        r"[?&]id=([^&]+)",
        r"/open\?id=([^&]+)",
        r"/thumbnail\?id=([^&]+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return unquote(match.group(1))
    return None


def slugify(value, max_len=80):
    value = "" if value is None else str(value)
    value = (
        unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    )
    value = re.sub(r"[^a-zA-Z0-9_-]+", "_", value).strip("_")
    value = re.sub(r"_+", "_", value)
    return value[:max_len] or "image"


def infer_extension(content_type: str, fallback=".jpg"):
    if not content_type:
        return fallback
    content_type = content_type.split(";")[0].strip().lower()
    ext = mimetypes.guess_extension(content_type)
    if ext == ".jpe":
        ext = ".jpg"
    return ext or fallback


def is_image_response(response: requests.Response):
    content_type = response.headers.get("content-type", "").lower()
    if content_type.startswith("image/"):
        return True
    # Fallback por magic bytes.
    head = response.content[:16] if response.content else b""
    return (
        head.startswith(b"\xff\xd8\xff")  # jpg
        or head.startswith(b"\x89PNG")
        or head.startswith(b"GIF8")
        or head.startswith(b"RIFF")  # webp suele iniciar con RIFF
    )


def existing_file_for_base(output_dir: Path, filename_base: str):
    matches = list(output_dir.glob(filename_base + ".*"))
    return matches[0] if matches else None


def build_filename_base(row, row_number: int, file_id: str):
    # Usa columnas disponibles si existen.
    parts = [f"row_{row_number:04d}"]

    for col in ["Common Name", "Especie (Clasificación)", "Camera ID"]:
        if col in row and pd.notna(row[col]):
            parts.append(slugify(row[col], max_len=40))

    if file_id:
        parts.append(slugify(file_id[:12]))

    return "__".join(parts)


def save_response_image(response: requests.Response, output_base: Path):
    ext = infer_extension(response.headers.get("content-type", ""), fallback=".jpg")
    output_path = output_base.with_suffix(ext)

    with open(output_path, "wb") as f:
        f.write(response.content)

    return output_path

In [ ]:
def request_image(
    session: requests.Session, url: str, output_base: Path, method: str, timeout=60
):
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0 Safari/537.36",
        "Accept": "image/avif,image/webp,image/apng,image/svg+xml,image/*,*/*;q=0.8",
        "Referer": "https://drive.google.com/",
    }

    response = session.get(url, headers=headers, timeout=timeout, allow_redirects=True)

    if response.status_code == 200 and is_image_response(response):
        output_path = save_response_image(response, output_base)
        return {
            "ok": True,
            "method": method,
            "saved_path": str(output_path),
            "content_type": response.headers.get("content-type", ""),
            "source_url": url,
            "error": None,
        }

    return {
        "ok": False,
        "method": method,
        "saved_path": None,
        "content_type": response.headers.get("content-type", ""),
        "source_url": url,
        "error": f"status={response.status_code}, content_type={response.headers.get('content-type', '')}",
    }


def try_drive_direct_download(
    session: requests.Session, file_id: str, output_base: Path
):
    # Descarga directa clásica.
    direct_url = f"https://drive.google.com/uc?export=download&id={file_id}"
    result = request_image(session, direct_url, output_base, method="drive_direct")
    if result["ok"]:
        return result

    # Manejo de confirm token para archivos grandes si aplica.
    response = session.get(direct_url, allow_redirects=True)
    token = None
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            token = value
            break

    if token:
        confirm_url = (
            f"https://drive.google.com/uc?export=download&confirm={token}&id={file_id}"
        )
        result = request_image(
            session, confirm_url, output_base, method="drive_direct_confirm"
        )
        if result["ok"]:
            return result

    return result


def try_drive_thumbnail(session: requests.Session, file_id: str, output_base: Path):
    last_result = None
    for size in PREFERRED_THUMBNAIL_SIZES:
        thumb_url = f"https://drive.google.com/thumbnail?id={file_id}&sz=w{size}"
        result = request_image(
            session, thumb_url, output_base, method=f"drive_thumbnail_w{size}"
        )
        last_result = result
        if result["ok"]:
            return result
    return last_result

In [ ]:
def extract_image_urls_from_html(html: str):
    soup = BeautifulSoup(html, "html.parser")
    candidates = []

    # Metatags útiles.
    for selector in [
        {"property": "og:image"},
        {"name": "twitter:image"},
    ]:
        tag = soup.find("meta", attrs=selector)
        if tag and tag.get("content"):
            candidates.append(tag["content"])

    # Imágenes directas en HTML.
    for img in soup.find_all("img"):
        src = img.get("src") or img.get("data-src")
        if src:
            candidates.append(src)

    # URLs embebidas en scripts o HTML.
    patterns = [
        r"https://lh3\.googleusercontent\.com/[^\"'\\<> ]+",
        r"https://drive\.google\.com/thumbnail\?[^\"'\\<> ]+",
        r"https://[^\"'\\<> ]+googleusercontent\.com/[^\"'\\<> ]+",
    ]
    for pattern in patterns:
        candidates.extend(re.findall(pattern, html))

    # Limpiar y quitar duplicados.
    clean = []
    seen = set()
    for url in candidates:
        url = url.replace("\\u003d", "=").replace("\\u0026", "&")
        url = url.replace("&amp;", "&")
        if url.startswith("//"):
            url = "https:" + url
        if url.startswith("http") and url not in seen:
            seen.add(url)
            clean.append(url)

    return clean


def try_scrape_html(session: requests.Session, page_url: str, output_base: Path):
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }
    response = session.get(page_url, headers=headers, timeout=60, allow_redirects=True)
    if response.status_code != 200:
        return {
            "ok": False,
            "method": "scrape_html",
            "saved_path": None,
            "content_type": response.headers.get("content-type", ""),
            "source_url": page_url,
            "error": f"page_status={response.status_code}",
        }

    candidates = extract_image_urls_from_html(response.text)
    last_result = None
    for candidate_url in candidates:
        result = request_image(
            session, candidate_url, output_base, method="scrape_html_candidate"
        )
        last_result = result
        if result["ok"]:
            return result

    return last_result or {
        "ok": False,
        "method": "scrape_html",
        "saved_path": None,
        "content_type": response.headers.get("content-type", ""),
        "source_url": page_url,
        "error": "No se encontraron URLs de imagen en HTML estático.",
    }

## 5. Fallback con Playwright para inspeccionar la preview

Este método abre el link con Chromium headless y busca URLs de imágenes cargadas dinámicamente por Google Drive. 

Es similar a inspeccionar la página manualmente, pero automatizado.

In [ ]:
async def get_preview_urls_with_playwright(page, page_url: str):
    await page.goto(page_url, wait_until="networkidle", timeout=BROWSER_TIMEOUT_MS)
    await page.wait_for_timeout(2500)

    urls = []

    # Metatags.
    try:
        meta_urls = await page.evaluate("""
            () => Array.from(document.querySelectorAll('meta[property="og:image"], meta[name="twitter:image"]'))
                .map(m => m.content)
                .filter(Boolean)
        """)
        urls.extend(meta_urls)
    except Exception:
        pass

    # Imágenes renderizadas.
    try:
        img_urls = await page.evaluate("""
            () => Array.from(document.images)
                .map(img => img.currentSrc || img.src || img.getAttribute('data-src'))
                .filter(Boolean)
        """)
        urls.extend(img_urls)
    except Exception:
        pass

    # Buscar también en el HTML final.
    try:
        html = await page.content()
        urls.extend(extract_image_urls_from_html(html))
    except Exception:
        pass

    # Filtrar candidatos probables de preview.
    filtered = []
    seen = set()
    keywords = [
        "googleusercontent",
        "thumbnail",
        "drive-viewer",
        "lh3.googleusercontent",
    ]
    blocked = ["logo", "favicon", "profile", "avatar"]

    for url in urls:
        if not isinstance(url, str):
            continue
        url = url.replace("&amp;", "&")
        if url.startswith("//"):
            url = "https:" + url
        url_low = url.lower()
        if not url.startswith("http"):
            continue
        if any(b in url_low for b in blocked):
            continue
        if any(k in url_low for k in keywords) and url not in seen:
            seen.add(url)
            filtered.append(url)

    return filtered


async def try_playwright_preview(
    page, session: requests.Session, page_url: str, output_base: Path
):
    try:
        candidates = await get_preview_urls_with_playwright(page, page_url)
    except Exception as e:
        return {
            "ok": False,
            "method": "playwright_preview",
            "saved_path": None,
            "content_type": None,
            "source_url": page_url,
            "error": f"Playwright error: {repr(e)}",
        }

    last_result = None
    for candidate_url in candidates:
        result = request_image(
            session, candidate_url, output_base, method="playwright_preview_candidate"
        )
        last_result = result
        if result["ok"]:
            return result

    return last_result or {
        "ok": False,
        "method": "playwright_preview",
        "saved_path": None,
        "content_type": None,
        "source_url": page_url,
        "error": "No se encontraron URLs descargables desde la preview.",
    }

## 6. Descargar imágenes desde el Excel

La celda siguiente procesa fila por fila y guarda un CSV incremental con el resultado.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq \
  libatk1.0-0 \
  libatk-bridge2.0-0 \
  libcups2 \
  libxkbcommon0 \
  libxcomposite1 \
  libxdamage1 \
  libxfixes3 \
  libxrandr2 \
  libgbm1 \
  libpango-1.0-0 \
  libcairo2 \
  libnss3 \
  libnspr4 \
  libasound2

!python -m playwright install --with-deps chromium

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../00-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../01-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../02-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../03-gsettings-desktop-schemas_42.0-1ubuntu1_all.deb ...
Unpacking gsettings-desktop-schemas (42.0-1ubuntu1) ...
Selecting previously unselected

In [ ]:
def normalize_result(base_info: dict, result: dict):
    out = dict(base_info)
    out.update(result or {})
    out["status"] = "downloaded" if out.get("ok") else "failed"
    return out


async def download_images_from_excel(work_df: pd.DataFrame):
    from playwright.async_api import async_playwright

    manifest_path = OUTPUT_DIR / "download_manifest.csv"
    results = []
    session = requests.Session()

    playwright = None
    browser = None
    page = None

    if USE_PLAYWRIGHT_FALLBACK:
        playwright = await async_playwright().start()
        browser = await playwright.chromium.launch(
            headless=HEADLESS_BROWSER,
            args=["--no-sandbox", "--disable-dev-shm-usage"],
        )
        page = await browser.new_page()

    try:
        iterator = tqdm(
            work_df.iterrows(), total=len(work_df), desc="Descargando imágenes"
        )
        for local_idx, row in iterator:
            original_row_number = int(local_idx) + 2  # +2 por header Excel y base 0
            link = str(row[LINK_COLUMN]).strip()
            file_id = extract_drive_file_id(link)
            filename_base = build_filename_base(
                row, original_row_number, file_id or "no_id"
            )
            output_base = OUTPUT_DIR / filename_base

            base_info = {
                "excel_row": original_row_number,
                "link": link,
                "file_id": file_id,
                "filename_base": filename_base,
            }

            if not file_id:
                results.append(
                    normalize_result(
                        base_info,
                        {
                            "ok": False,
                            "method": "extract_file_id",
                            "saved_path": None,
                            "content_type": None,
                            "source_url": link,
                            "error": "No se pudo extraer file_id desde el link.",
                        },
                    )
                )
                pd.DataFrame(results).to_csv(manifest_path, index=False)
                continue

            existing = existing_file_for_base(OUTPUT_DIR, filename_base)
            if existing is not None and not OVERWRITE:
                results.append(
                    normalize_result(
                        base_info,
                        {
                            "ok": True,
                            "method": "skipped_exists",
                            "saved_path": str(existing),
                            "content_type": None,
                            "source_url": link,
                            "error": None,
                        },
                    )
                )
                pd.DataFrame(results).to_csv(manifest_path, index=False)
                continue

            attempts = []

            # 1) Descarga directa.
            result = try_drive_direct_download(session, file_id, output_base)
            attempts.append(result)
            if not result["ok"]:
                # 2) Thumbnail de Google Drive.
                result = try_drive_thumbnail(session, file_id, output_base)
                attempts.append(result)

            if not result["ok"]:
                # 3) HTML estático.
                result = try_scrape_html(session, link, output_base)
                attempts.append(result)

            if not result["ok"] and USE_PLAYWRIGHT_FALLBACK and page is not None:
                # 4) Browser real para inspeccionar preview.
                result = await try_playwright_preview(page, session, link, output_base)
                attempts.append(result)

            if result["ok"]:
                final = normalize_result(base_info, result)
            else:
                errors = " | ".join(
                    f"{a.get('method')}: {a.get('error')}" for a in attempts if a
                )
                final = normalize_result(
                    base_info,
                    {
                        "ok": False,
                        "method": "all_methods_failed",
                        "saved_path": None,
                        "content_type": result.get("content_type") if result else None,
                        "source_url": link,
                        "error": errors,
                    },
                )

            results.append(final)
            pd.DataFrame(results).to_csv(manifest_path, index=False)
            time.sleep(REQUEST_SLEEP_SEC)

    finally:
        if browser is not None:
            await browser.close()
        if playwright is not None:
            await playwright.stop()

    manifest_df = pd.DataFrame(results)
    manifest_df.to_csv(manifest_path, index=False)
    return manifest_df


manifest_df = await download_images_from_excel(work_df)
print("Descarga finalizada.")
print("Manifest:", OUTPUT_DIR / "download_manifest.csv")
display(manifest_df.head(20))
print(manifest_df["status"].value_counts(dropna=False))

Descargando imágenes:   0%|          | 0/77 [00:00<?, ?it/s]

Descarga finalizada.
Manifest: /content/drive/MyDrive/ECHO/Metadata/downloaded_eval_images/download_manifest.csv


,excel_row,link,file_id,filename_base,ok,method,saved_path,content_type,source_url,error,status
0,2,https://drive.google.com/file/d/1IZAOIR8S5BK35...,1IZAOIR8S5BK35plLNiy0h3eHzE_w3pKC,row_0002__CAM6__1IZAOIR8S5BK,True,drive_thumbnail_w4096,/content/drive/MyDrive/ECHO/Metadata/downloade...,image/jpeg,https://drive.google.com/thumbnail?id=1IZAOIR8...,None,downloaded
1,3,https://drive.google.com/file/d/1umhS6rmaguIBs...,1umhS6rmaguIBs3ptQJofalbuKLdEIhVk,row_0003__CAM1__1umhS6rmaguI,True,drive_direct,/content/drive/MyDrive/ECHO/Metadata/downloade...,image/jpeg,https://drive.google.com/uc?export=download&id...,None,downloaded
2,4,https://drive.google.com/file/d/1q6xDb6qiuxyRN...,1q6xDb6qiuxyRNYttl7tt0YW7Ot8LZyJW,row_0004__CAM6__1q6xDb6qiuxy,True,drive_thumbnail_w4096,/content/drive/MyDrive/ECHO/Metadata/downloade...,image/jpeg,https://drive.google.com/thumbnail?id=1q6xDb6q...,None,downloaded
3,5,https://drive.google.com/file/d/1lysghJxxHy-RF...,1lysghJxxHy-RFf4GJfmxx3LlUncDJhTY,row_0005__CAM2__1lysghJxxHy-,True,drive_thumbnail_w4096,/content/drive/MyDrive/ECHO/Metadata/downloade...,image/jpeg,https://drive.google.com/thumbnail?id=1lysghJx...,None,downloaded
4,6,https://drive.google.com/file/d/1ev8j9iS1w6VUX...,1ev8j9iS1w6VUXlUhWWCu6WGjs_9Jm11s,row_0006__CAM4__1ev8j9iS1w6V,True,drive_thumbnail_w4096,/content/drive/MyDrive/ECHO/Metadata/downloade...,image/jpeg,https://drive.google.com/thumbnail?id=1ev8j9iS...,None,downloaded
5,7,https://drive.google.com/file/d/1qBIsLQho2Um2M...,1qBIsLQho2Um2MUi6JR5it2m_c8NlnnSF,row_0007__CAM4__1qBIsLQho2Um,True,drive_thumbnail_w4096,/content/drive/MyDrive/ECHO/Metadata/downloade...,image/jpeg,https://drive.google.com/thumbnail?id=1qBIsLQh...,None,downloaded
6,8,https://drive.google.com/file/d/1pRY-4tjZxxVhJ...,1pRY-4tjZxxVhJbVKNDX8o3U1w9R2xvU7,row_0008__CAM5__1pRY-4tjZxxV,True,drive_thumbnail_w4096,/content/drive/MyDrive/ECHO/Metadata/downloade...,image/jpeg,https://drive.google.com/thumbnail?id=1pRY-4tj...,None,downloaded
7,9,https://drive.google.com/file/d/16WNwViLSHM40S...,16WNwViLSHM40SDF_i05S5mrgnDwoURK-,row_0009__CAM4__16WNwViLSHM4,True,drive_thumbnail_w4096,/content/drive/MyDrive/ECHO/Metadata/downloade...,image/jpeg,https://drive.google.com/thumbnail?id=16WNwViL...,None,downloaded
8,10,https://drive.google.com/file/d/1vV7dw1slyKuJM...,1vV7dw1slyKuJMq-qjdVczFAZ38Klci9z,row_0010__CAM1__1vV7dw1slyKu,True,drive_thumbnail_w4096,/content/drive/MyDrive/ECHO/Metadata/downloade...,image/jpeg,https://drive.google.com/thumbnail?id=1vV7dw1s...,None,downloaded
9,11,https://drive.google.com/file/d/1D5o_S1zun_38R...,1D5o_S1zun_38R1SJGnbfLY_1MSCpje6b,row_0011__CAM1__1D5o_S1zun_3,True,drive_direct,/content/drive/MyDrive/ECHO/Metadata/downloade...,image/jpeg,https://drive.google.com/uc?export=download&id...,None,downloaded


status
downloaded    77
Name: count, dtype: int64


## 7. Revisar fallos

In [ ]:
failed = manifest_df[manifest_df["status"] != "downloaded"].copy()
print("Fallidas:", len(failed))
if len(failed) > 0:
    display(failed[["excel_row", "file_id", "link", "method", "error"]].head(50))

Fallidas: 0


## 8. Crear ZIP con las imágenes descargadas


In [ ]:
import shutil

zip_output = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("ZIP creado:", zip_output)

ZIP creado: /content/drive/MyDrive/ECHO/Metadata/downloaded_eval_images.zip
